# Video Pipeline Details

This notebook goes into detail about the stages of the video pipeline in the base overlay and is written for people who want to create and integrate their own video IP. For most regular input and output use cases the high level wrappers of `HDMIIn` and `HDMIOut` should be used.

Both the input and output pipelines in the base overlay consist of four stages, an HDMI frontend, a colorspace converter, a pixel format converter, and the video DMA. For the input the stages are arranged Frontend -> Colorspace Converter -> Pixel Format -> VDMA with the order reversed for the output side. The aim of this notebook is to give you enough information to use each stage separately and be able to modify the pipeline for your own ends.

Before exploring the pipeline we'll import the entire pynq.lib.video module where all classes relating to the pipelines live. We'll also load the base overlay to serve as an example.

The following table shows the IP responsible for each stage in the base overlay which will be referenced throughout the rest of the notebook

|Stage             | Input IP                               | Output IP                          |
|------------------|:---------------------------------------|:-----------------------------------|
|Frontend (Timing) |`video/hdmi_in/frontend/vtc_in`         |`video/hdmi_out/frontend/vtc_out`   |
|Frontend (Other)  |`video/hdmi_in/frontend/axi_gpio_hdmiin`|`video/hdmi_out/frontend/axi_dynclk`|
|Colour Space      |`video/hdmi_in/color_convert`           |`video/hdmi_out/color_convert`      |
|Pixel Format      |`video/hdmi_in/pixel_pack`              |`video/hdmi_outpixel_unpack`        |
|VDMA              |`video/axi_vdma`                        |`video/axi_vdam`                    |

## HDMI Frontend

The HDMI frontend modules wrap all of the clock and timing logic. The HDMI input frontend can be used independently from the rest of the pipeline by accessing its driver from the base overlay.

Creating the device will signal to the computer that a monitor is connected. Starting the frontend will wait attempt to detect the video mode, blocking until a lock can be achieved. Once the frontend is started the video mode will be available.

The HDMI output frontend can be accessed in a similar way.

and the mode must be set prior to starting the output. In this case we are just going to use the same mode as the input.

Note that nothing will be displayed on the screen as no video data is currently being send.

## Colorspace conversion

The colorspace converter operates on each pixel independently using a 3x4 matrix to transform the pixels. The converter is programmed with a list of twelve coefficients in the folling order:

|     |in1 |in2 |in3 | 1  |
|-----|----|----|----|----|
|out1 |c1  |c2  |c3  |c10 |
|out2 |c4  |c5  |c6  |c11 |
|out3 |c7  |c8  |c9  |c12 |

Each coefficient should be a floating point number between -2 and +2.

The pixels to and from the HDMI frontends are in BGR order so a list of coefficients to convert from the input format to RGB would be:

    [0, 0, 1,
     0, 1, 0,
     1, 0, 0,
     0, 0, 0]

reversing the order of the pixels and not adding any bias.

The driver for the colorspace converters has a single property that contains the list of coefficients.

## Pixel format conversion

The pixel format converters convert between the 24-bit signal used by the HDMI frontends and the colorspace converters to either an 8, 24, or 32 bit signal. 24-bit mode passes the input straight through, 32-bit pads the additional pixel with 0 and 8-bit mode selects the first channel in the pixel. This is exposed by a single property to set or get the number of bits.

## Video DMA

The final element in the pipeline is the video DMA which transfers video frames to and from memory. The VDMA consists of two channels, one for each direction which operate completely independently. To use a channel its mode must be set prior to start being called. After the DMA is started `readframe` and `writeframe` transfer frames. Frames are only transferred once with the call blocking if necessary. `asyncio` coroutines are available as `readframe_async` and `writeframe_async` which yield instead of blocking. A frame of the size of the output can be retrieved from the VDMA by calling `writechannel.newframe()`. This frame is not guaranteed to be initialised to blank so should be completely written before being handed back.

In this case, because we are only using 8 bits per pixel, only the red channel is read and displayed.

The two channels can be tied together which will ensure that the input is always mirrored to the output

In [1]:
# Full restart
# vdma.readchannel.stop()
# vdma.writechannel.stop()
from pynq.overlays.base import BaseOverlay
from pynq import MMIO
import os, datetime
from pynq.lib.video import *
from IPython.display import clear_output
import time

from pynq.overlays.base import BaseOverlay
base = BaseOverlay("base.bit")

# Redo everything from scratch
hdmiin_frontend = base.video.hdmi_in.frontend
hdmiin_frontend.start()

hdmiout_frontend = base.video.hdmi_out.frontend
hdmiout_frontend.mode = VideoMode(width=1920, height=1080, bits_per_pixel=24, fps=240)
hdmiout_frontend.start()

colorspace_in  = base.video.hdmi_in.color_convert
colorspace_out = base.video.hdmi_out.color_convert
bgr2rgb = [0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0]
colorspace_in.colorspace  = bgr2rgb
colorspace_out.colorspace = bgr2rgb

pixel_in  = base.video.hdmi_in.pixel_pack
pixel_out = base.video.hdmi_out.pixel_unpack
pixel_in.bits_per_pixel  = 8
pixel_out.bits_per_pixel = 8

vdma = base.video.axi_vdma
in_width        = hdmiin_frontend.mode.width
print(in_width)
in_height       = hdmiin_frontend.mode.height
out_width       = 1920
out_height      = 1080
bytes_per_pixel = 1
stride          = in_width * bytes_per_pixel

vdma.writechannel.mode = VideoMode(in_width,  in_height, 8)
vdma.readchannel.mode  = VideoMode(out_width, out_height, 8)



vdma.writechannel.start()
frame     = vdma.writechannel._cache.getframe()
frame.flush()
base_addr = frame.physical_address
print(f"fresh base_addr: {hex(base_addr)}")
vdma.readchannel.start()

vdma.writechannel._mmio.write(0xAC, base_addr)
vdma.writechannel._mmio.write(0xA8, in_width  * bytes_per_pixel)
vdma.writechannel._mmio.write(0xA4, in_width  * bytes_per_pixel)
vdma.writechannel._mmio.write(0xA0, in_height)



# vdma.readchannel._mmio.write(0x5C, base_addr)
# vdma.readchannel._mmio.write(0x58, stride)
# vdma.readchannel._mmio.write(0x54, out_width  * bytes_per_pixel)
# vdma.readchannel._mmio.write(0x50, out_height)

ctrl  = vdma.readchannel._mmio.read(0x30)
ctrl &= ~0xFF00
ctrl |= (0x01 << 8) | (1 << 12)
vdma.readchannel._mmio.write(0x30, ctrl)

time.sleep(1.0)
print(f"MM2S status: {hex(vdma.readchannel._mmio.read(0x34))}")
print(f"IOC:    {(vdma.readchannel._mmio.read(0x34) >> 12) & 1}")
print(f"IntErr: {(vdma.readchannel._mmio.read(0x34) >> 4)  & 1}")

# Check raw_cnt on fresh start
CTRL_REG       = 0x00
BASE_ADDR_REG  = 0x04
OUT_WIDTH_REG  = 0x08
OUT_HEIGHT_REG = 0x0C
BPP_REG        = 0x10
STRIDE_REG     = 0x14
VDMA_ADDR_REG  = 0x18
STATUS_REG     = 0x1C
TRIG_REG       = 0x20

qs_addr        = base.ip_dict['quadrant_switcher_1']['phys_addr']
vdma_phys_addr = base.ip_dict['video/axi_vdma']['phys_addr']
qs = MMIO(qs_addr, 0x1000)


# # Check counters before enabling QS
# print("\n=== Counters before QS enable ===")
# for i in range(5):
#     q, s, fd, raw_cnt, frame_cnt = read_qs()
#     print(f"  raw_cnt={raw_cnt} frame_cnt={frame_cnt} Q={q}")
#     time.sleep(0.2)

# Enable QS
qs.write(CTRL_REG,       0)
qs.write(BASE_ADDR_REG,  base_addr)
qs.write(OUT_WIDTH_REG,  out_width)
qs.write(OUT_HEIGHT_REG, out_height)
qs.write(BPP_REG,        bytes_per_pixel)
qs.write(STRIDE_REG,     stride)
qs.write(VDMA_ADDR_REG,  vdma_phys_addr)
qs.write(CTRL_REG, 1)
print("\nQS enabled")

print(f"in_width:   {in_width}")
print(f"in_height:  {in_height}")  
print(f"out_width:  {out_width}")
print(f"out_height: {out_height}")
print(f"stride:     {stride}")
print(f"stride reg: {qs.read(STRIDE_REG)}")
print(f"Q0: {hex(base_addr)}")
print(f"Q1: {hex(base_addr + out_width * bytes_per_pixel)}")
print(f"Q2: {hex(base_addr + out_height * stride)}")
print(f"Q3: {hex(base_addr + out_height * stride + out_width * bytes_per_pixel)}")
    
# try:
#     while True:
#         q, s, fd, raw_c, frm_c = read_qs()
#         mm2s = vdma.mmio.read(0x04)
#         ioc  = (mm2s >> 12) & 1

#         clear_output(wait=True)   # <-- clears the cell output, then reprints below

# #         print(f"Q={q}  state={s}  fd_pending={fd}")
# #         print(f"raw_cnt={raw_c}  frame_cnt={frm_c}")
# #         print(f"MM2S_SR=0x{mm2s:08x}  IOC={ioc}")
   
   
# except KeyboardInterrupt:
#     print("stopped")

3840
fresh base_addr: 0x60b00000
MM2S status: 0x4d000
IOC:    1
IntErr: 0

QS enabled
in_width:   3840
in_height:  2160
out_width:  1920
out_height: 1080
stride:     3840
stride reg: 3840
Q0: 0x60b00000
Q1: 0x60b00780
Q2: 0x60ef4800
Q3: 0x60ef4f80


In [10]:
print(f"MM2S ctrl:   {hex(vdma.readchannel._mmio.read(0x30))}")
print(f"MM2S status: {hex(vdma.readchannel._mmio.read(0x34))}")
print(f"MM2S addr:   {hex(vdma.readchannel._mmio.read(0x5C))}")
print(f"MM2S hsize:  {hex(vdma.readchannel._mmio.read(0x54))}")
print(f"MM2S vsize:  {hex(vdma.readchannel._mmio.read(0x50))}")
print(f"MM2S stride: {hex(vdma.readchannel._mmio.read(0x58))}")

MM2S ctrl:   0x10082
MM2S status: 0x10001
MM2S addr:   0x0
MM2S hsize:  0x0
MM2S vsize:  0x0
MM2S stride: 0x0


### Frame Ownership

The VDMA driver has a strict method of frame ownership. Any frames returned by `readframe` or `newframe` are owned by the user and should be destroyed by the user when no longer needed by calling `frame.freebuffer()`. Frames handed back to the VDMA with `writeframe` are no longer owned by the user and should not be touched - the data may disappear at any time.

## Cleaning up

It is vital to stop the VDMA before reprogramming the bitstream otherwise the memory system of the chip can be placed into an undefined state. If the monitor does not power on when starting the VDMA this is the likely cause.

In [ ]:
vdma.readchannel.stop()
vdma.writechannel.stop()